In [0]:
base_path = "/Volumes/workspace/default/my_volume/flight_project"

bronze_flights_path = f"{base_path}/bronze/flights"
bronze_weather_path = f"{base_path}/bronze/weather"
silver_path         = f"{base_path}/silver/flights"

print("Paths set!")

Paths set!


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df_flights = spark.read.format("delta").load(bronze_flights_path)
df_weather = spark.read.format("delta").load(bronze_weather_path)

# Basic cleaning
df_clean = df_flights.filter(
    col("on_ground") == False          # sirf airborne flights
).withColumn(
    "callsign", trim(col("callsign"))  # whitespace remove
).withColumn(
    "airline_code",                    # pehle 3 chars = airline code
    substring(col("callsign"), 1, 3)
)

print(f"Airborne flights: {df_clean.count()}")
df_clean.select("icao24", "callsign", "airline_code", 
                "velocity", "baro_altitude").show(5)

Airborne flights: 12073
+------+--------+------------+--------+-------------+
|icao24|callsign|airline_code|velocity|baro_altitude|
+------+--------+------------+--------+-------------+
|39de4f| TVF98NQ|         TVF|  240.83|      11582.4|
|a89ea5|  N6545H|         N65|   57.85|       624.84|
|511141|   ESETI|         ESE|   91.39|      1965.96|
|3ffc29|   DMFSS|         DMF|   50.48|       891.54|
|ae1fa6| TALON57|         TAL|    42.7|      1630.68|
+------+--------+------------+--------+-------------+
only showing top 5 rows


In [0]:
df_delay = df_clean.withColumn(
    "flight_phase",
    when(col("baro_altitude") < 1000, "LANDING")
    .when(col("baro_altitude") < 3000, "APPROACH")
    .when(col("baro_altitude") > 9000, "CRUISE")
    .otherwise("CLIMB_DESCENT")
).withColumn(
    "delay_score",
    (
        when((col("flight_phase") == "APPROACH") & 
             (col("vertical_rate") > -2), 40).otherwise(0) +
        when((col("flight_phase") == "CRUISE") & 
             (col("velocity") < 150), 30).otherwise(0) +
        when(col("velocity") < 100, 20).otherwise(0)
    )
).withColumn(
    "is_delayed",
    when(col("delay_score") >= 30, lit(True))
    .otherwise(lit(False))
).withColumn(
    "delay_minutes",
    when(col("delay_score") >= 50, lit(45))
    .when(col("delay_score") >= 30, lit(20))
    .otherwise(lit(0))
)

print("Flight phases:")
df_delay.groupBy("flight_phase").count().orderBy("count", ascending=False).show()

print("Delayed flights:")
df_delay.groupBy("is_delayed").count().show()

print("Delay score distribution:")
df_delay.groupBy("delay_score").count().orderBy("delay_score").show()

Flight phases:
+-------------+-----+
| flight_phase|count|
+-------------+-----+
|       CRUISE| 4812|
|CLIMB_DESCENT| 2780|
|      LANDING| 2399|
|     APPROACH| 2082|
+-------------+-----+

Delayed flights:
+----------+-----+
|is_delayed|count|
+----------+-----+
|      true| 1508|
|     false|10565|
+----------+-----+

Delay score distribution:
+-----------+-----+
|delay_score|count|
+-----------+-----+
|          0| 7832|
|         20| 2733|
|         30|   23|
|         40|  359|
|         50|    9|
|         60| 1117|
+-----------+-----+



In [0]:
dbutils.fs.ls("/Volumes/workspace/default/my_volume/")

[FileInfo(path='dbfs:/Volumes/workspace/default/my_volume/airports.dat.txt', name='airports.dat.txt', size=1127225, modificationTime=1782473927000),
 FileInfo(path='dbfs:/Volumes/workspace/default/my_volume/checkpoint/', name='checkpoint/', size=0, modificationTime=1782473960514),
 FileInfo(path='dbfs:/Volumes/workspace/default/my_volume/checkpoints/', name='checkpoints/', size=0, modificationTime=1782473960514),
 FileInfo(path='dbfs:/Volumes/workspace/default/my_volume/flight_project/', name='flight_project/', size=0, modificationTime=1782473960514),
 FileInfo(path='dbfs:/Volumes/workspace/default/my_volume/orders_delta/', name='orders_delta/', size=0, modificationTime=1782473960514),
 FileInfo(path='dbfs:/Volumes/workspace/default/my_volume/orders_stream/', name='orders_stream/', size=0, modificationTime=1782473960514),
 FileInfo(path='dbfs:/Volumes/workspace/default/my_volume/raw_orders/', name='raw_orders/', size=0, modificationTime=1782473960514),
 FileInfo(path='dbfs:/Volumes/wor

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
import math


# Step 1 — Load airports
airport_schema = StructType([
    StructField("airport_id", StringType()),
    StructField("name",       StringType()),
    StructField("city",       StringType()),
    StructField("country",    StringType()),
    StructField("iata_code",  StringType()),
    StructField("icao_code",  StringType()),
    StructField("latitude",   DoubleType()),
    StructField("longitude",  DoubleType()),
    StructField("altitude",   StringType()),
    StructField("timezone",   StringType()),
    StructField("dst",        StringType()),
    StructField("tz",         StringType()),
    StructField("type",       StringType()),
    StructField("source",     StringType())
])

airports_df = spark.read.csv(
    "/Volumes/workspace/default/my_volume/airports.dat.txt",
    schema=airport_schema,
    header=False
).filter(
    col("iata_code") != "\\N"
).select("iata_code", "latitude", "longitude")

print(f"Total airports loaded: {airports_df.count()}")
airports_df.show(5)

Total airports loaded: 6072
+---------+------------------+------------------+
|iata_code|          latitude|         longitude|
+---------+------------------+------------------+
|      GKA|-6.081689834590001|     145.391998291|
|      MAG|    -5.20707988739|     145.789001465|
|      HGU|-5.826789855957031|144.29600524902344|
|      LAE|         -6.569803|        146.725977|
|      POM|-9.443380355834961|147.22000122070312|
+---------+------------------+------------------+
only showing top 5 rows


In [0]:
# Step 2 — Haversine UDF
@udf(DoubleType())
def haversine(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + \
        math.cos(math.radians(lat1)) * \
        math.cos(math.radians(lat2)) * \
        math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# Airports ko alias do
airports_aliased = airports_df.select(
    col("iata_code"),
    col("latitude").alias("airport_lat"),
    col("longitude").alias("airport_lon")
)

# Step 3 — Cross join + distance calculate
df_with_distance = df_delay.join(
    airports_aliased, how="cross"
).withColumn(
    "distance_km",
    haversine(
        col("latitude"),      # flight latitude
        col("longitude"),     # flight longitude
        col("airport_lat"),   # airport latitude
        col("airport_lon")    # airport longitude
    )
)

# Step 4 — Nearest airport per flight
from pyspark.sql.window import Window

window = Window.partitionBy("icao24").orderBy("distance_km")

df_enriched = df_with_distance.withColumn(
    "rank", row_number().over(window)
).filter(col("rank") == 1).withColumn(
    "nearest_airport",
    coalesce(col("iata_code"), lit("UNKNOWN"))
).drop("rank", "distance_km", "iata_code", "airport_lat", "airport_lon")

print(f"Total enriched flights: {df_enriched.count()}")
df_enriched.select(
    "icao24", "callsign", "latitude",
    "longitude", "nearest_airport"
).show(5)

Total enriched flights: 12073
+------+--------+--------+---------+---------------+
|icao24|callsign|latitude|longitude|nearest_airport|
+------+--------+--------+---------+---------------+
|008081| LNK929T|-32.7171|  20.0142|            ROD|
|0080f5|   ZSRMP|-33.8117|  18.9258|            CPT|
|008112|   ZSMCW|  -25.94|  27.9249|            HLA|
|0081ef|  SAA077|-26.3245|  28.1802|            QRA|
|0083cf|   ZSFZI| -25.482|  28.2791|            PRY|
+------+--------+--------+---------+---------------+
only showing top 5 rows


In [0]:
# Weather DataFrame rename karo pehle
df_weather_renamed = df_weather.select(
    col("airport_code").alias("w_airport_code"),
    col("temperature"),
    col("weather_main"),
    col("weather_desc"),
    col("wind_speed"),
    col("visibility")
)

df_silver = df_enriched.join(
    df_weather_renamed,
    df_enriched["nearest_airport"] == df_weather_renamed["w_airport_code"],
    "left"
).drop("w_airport_code"
).withColumn(
    "delay_reason",
    when(
        (col("is_delayed") == True) &
        (col("weather_main").isin("Thunderstorm", "Rain", "Snow", "Fog")),
        lit("WEATHER")
    ).when(
        (col("is_delayed") == True) &
        (col("wind_speed") > 10),
        lit("HIGH_WINDS")
    ).when(
        col("is_delayed") == True,
        lit("UNKNOWN")
    ).otherwise(lit("ON_TIME"))
).withColumn(
    "processed_at", current_timestamp()
)

print("Delay reasons:")
df_silver.groupBy("delay_reason").count() \
    .orderBy("count", ascending=False).show()

print(f"Total Silver rows: {df_silver.count()}")

Delay reasons:
+------------+-----+
|delay_reason|count|
+------------+-----+
|     ON_TIME|10565|
|     UNKNOWN| 1508|
+------------+-----+

Total Silver rows: 12073


In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print(f"✅ Silver table saved!")
print(f"Total rows: {df_silver.count()}")

# Quick summary
df_silver.select(
    "icao24", "callsign", "nearest_airport",
    "flight_phase", "delay_score", "is_delayed",
    "delay_minutes", "delay_reason",
    "weather_main", "temperature"
).show(5)

✅ Silver table saved!
Total rows: 12073
+------+--------+---------------+------------+-----------+----------+-------------+------------+------------+-----------+
|icao24|callsign|nearest_airport|flight_phase|delay_score|is_delayed|delay_minutes|delay_reason|weather_main|temperature|
+------+--------+---------------+------------+-----------+----------+-------------+------------+------------+-----------+
|008081| LNK929T|            ROD|      CRUISE|          0|     false|            0|     ON_TIME|        NULL|       NULL|
|0080f5|   ZSRMP|            CPT|     LANDING|         20|     false|            0|     ON_TIME|        NULL|       NULL|
|008112|   ZSMCW|            HLA|    APPROACH|         60|      true|           45|     UNKNOWN|        NULL|       NULL|
|0081ef|  SAA077|            QRA|    APPROACH|          0|     false|            0|     ON_TIME|        NULL|       NULL|
|0083cf|   ZSFZI|            PRY|    APPROACH|         20|     false|            0|     ON_TIME|        NU